# Notebook 07 — Career Pipeline
**Mission Veteran AI · Veteran Career Navigator · **

Runs the complete end-to-end pipeline in a single notebook:

| Step | Tool | Notebook |
|------|------|----------|
| 1 | `translate_bullet` (Bullet Translation) | 01 |
| 2 | `map_military_skills` (Skill Mapping) | 02 |
| 3 | `analyze_skills_gap` (Skill Mapping) | 02 |
| 4 | `analyze_job_description` () | 03 |
| 5 | `build_resume_draft` () | 04 |
| 6 | `generate_interview_prep` () | — |
| 7 | `evaluate_export_gate` () | 06 |

**Prerequisites**
- All previous notebooks completed  
- `.env` with `WATSONX_API_KEY`, `WATSONX_URL`, `WATSONX_PROJECT_ID`  
- All Bullet Translation–3 tools importable from `/tools/`

In [ ]:
%pip install -q ibm-watsonx-ai python-dotenv

In [ ]:
import json, os, re, sys
from dotenv import load_dotenv

load_dotenv()
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

API_KEY    = os.environ["WATSONX_API_KEY"]
WX_URL     = os.environ.get("WATSONX_URL", "https://us-south.ml.cloud.ibm.com")
PROJECT_ID = os.environ["WATSONX_PROJECT_ID"]
print("Credentials loaded ✓")


## 0 · Load all tool functions (bypass @tool decorator for direct testing)

In [ ]:
# Import toolkit functions directly (no importlib loader needed)
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from toolkits.bullet_translator.translate_bullet  import translate_bullet
from toolkits.skill_mapper.map_military_skills    import map_military_skills
from toolkits.gap_analyzer.analyze_skills_gap     import analyze_skills_gap
from toolkits.plan_builder.build_plan             import (
    analyze_job_description,
    build_resume_draft,
    evaluate_export_gate,
)

print('All toolkit functions loaded \u2713')


## 1 · Veteran profile inputs

In [ ]:
PROFILE = {
    'service_branch':   'Army',
    'specialty_code':   '92A',
    'target_career_path': 'Logistics',
    'service_summary': (
        'Managed automated supply logistics for a 500-person organization over 4 years, '
        'overseeing a $2.1M annual inventory. Led a 12-member team, improved on-time '
        'delivery from 82% to 96%, and implemented a cycle-count program that reduced '
        'shrinkage by 18%.'
    ),
    'bullets': [
        'Supervised supply accountability procedures for 3,200 line items valued at $2.1M with 98.7% accuracy.',
        'Led team of 12 in automated supply operations, achieving top unit readiness rating for 3 consecutive cycles.',
        'Implemented cycle count program reducing inventory shrinkage by 18% within first 90 days.',
    ],
    'job_description': """
        Supply Chain Operations Manager — Mid-level (3-5 yrs)
        Required: supply chain coordination, inventory management, vendor management,
        budget oversight, ERP/WMS proficiency, cross-functional team leadership.
        Keywords: supply chain, logistics, inventory, KPI, cross-functional, procurement.
    """,
    'target_role': 'Supply Chain Operations Manager',
}
print('Profile defined ✓')

## Step 1 · Translate bullets (Bullet Translation)

In [ ]:
translated = []
for bullet in PROFILE['bullets']:
    result = translate_bullet(
        bullet=bullet,
        target_role=PROFILE['target_career_path'],
    )
    if result.get('error'):
        print(f'Error: {result["error"]}')
    else:
        translated.append(result.get('translated_bullet', bullet))
        print(f'  \u2713  {result.get("translated_bullet", "")}')

print(f'\n{len(translated)} bullets translated \u2713')


## Step 2 · Map military skills (Skill Mapping)

In [ ]:
skill_result = map_military_skills(
    service_branch=PROFILE['service_branch'],
    specialty_code=PROFILE['specialty_code'],
    service_summary=PROFILE['service_summary'],
    target_career_path=PROFILE['target_career_path'],
)

if skill_result.get('error'):
    raise RuntimeError(skill_result['error'])

skills = skill_result.get('skills', [])
print(f'{len(skills)} skills mapped \u2713')
for s in skills:
    print(f"  {s.get('skill','')} ({s.get('category','')})") 


## Step 3 · Gap analysis (Skill Mapping)

In [ ]:
gap_result = analyze_skills_gap(
    veteran_skills_json=json.dumps(skills),
    target_career_path=PROFILE['target_career_path'],
    job_description=PROFILE['job_description'],
)

if gap_result.get('error'):
    raise RuntimeError(gap_result['error'])

print(f"Coverage: {gap_result.get('coverage_pct', 0)}%")
print(f"Covered: {len(gap_result.get('covered_skills', []))} skills")
print(f"Gaps:    {len(gap_result.get('gap_skills', []))} skills")


## Step 4 · Job description analysis ()

In [ ]:
job_result = analyze_job_description(
    job_description=PROFILE['job_description'],
)

if job_result.get('error'):
    raise RuntimeError(job_result['error'])

print(f"Role:      {job_result.get('role_level', '\u2014')}")
print(f"Industry:  {job_result.get('industry', '\u2014')}")
print(f"Required:  {len(job_result.get('required_skills', []))} skills")
print(f"Preferred: {len(job_result.get('preferred_skills', []))} skills")


## Step 5 · Build tagged resume draft ()

In [ ]:
resume_result = build_resume_draft(
    service_branch=PROFILE['service_branch'],
    specialty_code=PROFILE['specialty_code'],
    service_summary=PROFILE['service_summary'],
    translated_bullets_json=json.dumps(translated),
    skill_inventory_json=json.dumps(skills),
    job_analysis_json=json.dumps(job_result),
)

if resume_result.get('error'):
    raise RuntimeError(resume_result['error'])

print(f"EVIDENCED: {resume_result.get('evidenced_count', 0)}")
print(f"INFERRED:  {resume_result.get('inferred_count', 0)}")
print(f"MISSING:   {resume_result.get('missing_count', 0)}")
print(f"\nDisclaimer: {resume_result.get('disclaimer', '')}")


## Step 6 · Generate interview prep ()

In [ ]:
# NOTE: interview_prep has no toolkit equivalent yet.
# This step is skipped. To add it, implement a new @tool function in
# toolkits/plan_builder/build_plan.py and import it above.
prep_result = {'questions': [], 'missing_count': 0}
print('Interview prep step skipped (toolkit not yet implemented).')


## Step 7 · Evaluate export gate ()

In [ ]:
gate_result = evaluate_export_gate(
    inferred_count=resume_result.get('inferred_count', 0),
    missing_count=resume_result.get('missing_count', 0),
)

print(f'Gate status: {gate_result.get("gate_status", "?")}')
print(f'Message:     {gate_result.get("message", "")}')

holds = gate_result.get('holds', [])
if holds:
    print('\nHolds:')
    for h in holds:
        print(f'  \u26a0  {h}')
else:
    print('\n\u2705  No holds — export gate OPEN.')


## 8 · Pipeline summary

In [ ]:
print('═' * 60)
print('VETERAN CAREER NAVIGATOR — FULL PIPELINE SUMMARY')
print('═' * 60)
print(f'Bullets translated:      {len(translated)}')
print(f'Skills mapped:           {len(skills)}')
print(f'Skill coverage:          {gap_result.get("coverage_pct", 0)}%')
print(f'Job alignment:           {job_result.get("alignment", {}).get("alignment_pct", 0)}%')
print(f'Resume tag counts:       {resume_result.get("tag_counts", {})}')
print(f'Interview questions:     {len(prep_result.get("questions", []))}')
print(f'Gate status:             {gate_result.get("gate_status", "?")}')  
print(f'Gate holds:              {gate_result.get("blocked_count", 0)}')
print('═' * 60)

if gate_result.get('gate_status') == 'OPEN':
    print('✓ All clear. Document package is ready to export.')
else:
    print('✗ Export blocked. Resolve all gate holds before exporting.')